# NBRA (step3) workflow for CP2K/Libra made simple, and (hopefully) more accurate

In this tutorial, I am going to show how the NBRA calculations (time-overlap, vibronic Hamiltonian, etc.) can be done using the outputs of the TD-DFT calculations in CP2K. First, I remind how the relevant information can be parsed from the output files. Second, I will show the implementation of workflow. Third, I will show how to use one of the Libra's modules to do this calculations and how to control the truncation of the Slated determinants. 


## Table of contents
<a name="toc"></a>
1. [Extracting TD-DFT data from CP2K log files](#1)

2. [Libra implementation - with active space truncation](#2)


### A. Learning objectives

* To extract the key infromation from the output file of CP2K calculations (energies, CI vectors, etc.)
* Understand the indexing conventions for various variables and properties
* To automate the calculations of the CI time-overlaps, adiabatic Hamiltonians, and adiabatic vibronic Hamiltonians using CP2K
* To use the active space in CP2K/Libra workflow to reduce computational expenses 


### B. Use cases

* computing CI wavefunction time-overlaps with CP2K
* define Libra/CP2K interface Hamiltonian
* computing many-body (TD-DFT) NACs

### C. Functions

- `libra_py`
  - `citools`
    - `ci`
      - [`overlap`](#overlap-1)
  - `packages`
    - `cp2k`
      - `methods`
        - [`cp2k_compute_adi`](#cp2k_compute_adi-1)
        - [`read_cp2k_tddfpt_log_file`](#read_cp2k_tddfpt_log_file-1)


In [1]:
import os, time, copy, sys
import numpy as np
import scipy.sparse as sp

from libra_py import units
from libra_py import data_conv
import libra_py.packages.cp2k.methods as CP2K_methods
import libra_py.citools.ci as ci
from liblibra_core import *

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

Before starting, please unpack the data files:

In [2]:
#!tar -xf c20.tar.bz2

## 1. Extracting TD-DFT data from CP2K log files 
<a name="1"></a>
[Back to TOC](#toc)

Let's see how to extract the key information from excited state calculations from CK2K log files. 

This is done with the help of the `CP2K_methods.read_cp2k_tddfpt_log_file` function, which is recently revised, so many previous workflows (prior to **3/23/2026** may break down and are now considered obsolete!)

The function returns the general info with the key numbers, and the list of data elements - packaged in the same way as MOPAC/INDO-CI or DFTB+/TD-DFTB calculations in the recently added workflows, so we can take advantage of the recently added generalized workflows.
<a name="read_cp2k_tddfpt_log_file-1"></a>

In [3]:
help(CP2K_methods.read_cp2k_tddfpt_log_file)

Help on function read_cp2k_tddfpt_log_file in module libra_py.packages.cp2k.methods:

read_cp2k_tddfpt_log_file(params)
    Parse a CP2K TD-DFPT log file and extract excitation energies and CI expansion data.
    
    This function reads the output of a CP2K Time-Dependent Density Functional
    Perturbation Theory (TD-DFPT) calculation and reconstructs the excited-state
    wavefunctions in terms of single-particle (Kohn–Sham) excitations.
    
    In CP2K TD-DFPT, each excited state is represented as a linear combination of
    single excitations (Slater determinants) of the form:
    
        |Ψ_I⟩ = Σ_{i→a} C_{i→a}^{(I)} |Φ_i^a⟩
    
    where:
        - i denotes an occupied orbital
        - a denotes a virtual (unoccupied) orbital
        - C_{i→a}^{(I)} are CI-like expansion coefficients
        - |Φ_i^a⟩ is a singly-excited Slater determinant
    
    The function extracts:
        1. Excitation energies for each TD-DFPT state
        2. CI basis configurations (orbital transi

In [4]:
params_tddft = {'number_of_states': 11, 
                'tolerance': 0.0, 
                'logfile_name': 'tio2/all_logfiles/step_1000.log', 
                'isUKS': False, 
                }
info, data = CP2K_methods.read_cp2k_tddfpt_log_file(params_tddft)

In [5]:
print(info)

{'nocc': 24, 'nelec': 48, 'nao': 74, 'nmo': 74, 'nci': 10, 'min_occ': 13, 'max_occ': 24, 'min_vir': 25, 'max_vir': 36, 'nact': 74, 'actual_orbital_space': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74]}


The first argument of the `data` that was just extracted contains the excited states energies in eV.

In [6]:
data[0]

[0.14437359891220464,
 0.14985153063099482,
 0.15303075961927162,
 0.157557237881739,
 0.16686229833523208,
 0.17020653412223,
 0.1721370034177355,
 0.17645805005328727,
 0.17870273051339533,
 0.1818305097203337]

The second output contains the excited states configurations. Its length is `number_of_states` and each element of that contain a list which is formed from one or multiple lists, each representing a single-particle excitation. For example:

In [7]:
data[1]

[[[24, 25],
  [22, 26],
  [22, 25],
  [24, 27],
  [24, 26],
  [18, 26],
  [18, 25],
  [23, 25],
  [23, 26],
  [24, 34],
  [22, 27],
  [22, 36],
  [22, 32],
  [24, 28]],
 [[24, 26],
  [22, 25],
  [22, 26],
  [23, 25],
  [24, 25],
  [22, 27],
  [24, 27],
  [23, 27],
  [17, 26],
  [21, 26],
  [18, 25],
  [24, 32],
  [22, 34],
  [24, 28],
  [24, 36],
  [23, 26],
  [18, 26],
  [24, 30],
  [19, 26]],
 [[23, 25],
  [21, 26],
  [23, 27],
  [21, 25],
  [22, 25],
  [24, 26],
  [19, 26],
  [23, 26],
  [19, 25],
  [21, 30],
  [24, 28],
  [23, 28],
  [22, 27],
  [21, 27],
  [19, 32],
  [20, 31],
  [21, 32],
  [22, 26],
  [23, 34],
  [22, 28]],
 [[23, 26],
  [21, 25],
  [19, 25],
  [21, 27],
  [21, 26],
  [23, 25],
  [22, 26],
  [23, 32],
  [23, 27],
  [23, 30],
  [24, 29],
  [19, 26],
  [22, 28],
  [20, 30],
  [19, 27],
  [22, 25],
  [22, 27],
  [24, 25],
  [23, 28],
  [19, 31],
  [21, 28],
  [14, 26]],
 [[20, 25],
  [19, 25],
  [19, 26],
  [24, 28],
  [21, 25],
  [21, 26],
  [20, 26],
  [22, 28],


The third element of the `data` variable contains the configuration interaction coefficients related to each single-particle excitation in each excited states.

In [8]:
data[2]

[[-0.767561,
  -0.348362,
  0.292344,
  0.267854,
  0.231994,
  0.129112,
  -0.094824,
  -0.090934,
  0.088576,
  -0.083763,
  -0.064871,
  -0.05857,
  -0.054067,
  -0.052221],
 [-0.682742,
  -0.43614,
  -0.262745,
  0.256485,
  -0.251215,
  0.16214,
  0.121174,
  -0.111367,
  -0.11131,
  0.109796,
  0.097893,
  -0.095391,
  -0.071342,
  -0.070451,
  -0.069508,
  0.066891,
  0.061696,
  -0.056507,
  -0.052056],
 [-0.70504,
  -0.355546,
  0.314132,
  0.230779,
  -0.203583,
  -0.202846,
  0.188519,
  0.123218,
  -0.120975,
  -0.101993,
  0.099998,
  -0.065331,
  0.064846,
  -0.062559,
  0.057836,
  -0.056793,
  -0.056566,
  -0.055354,
  -0.055271,
  -0.050867],
 [0.741519,
  0.421698,
  -0.225953,
  -0.197854,
  0.140417,
  0.135918,
  0.118768,
  0.111836,
  -0.111822,
  0.110938,
  -0.108413,
  -0.089751,
  -0.089385,
  0.081729,
  0.071986,
  0.067255,
  -0.060396,
  0.058667,
  0.054931,
  0.05153,
  0.050488,
  0.050042],
 [0.506904,
  -0.400756,
  0.373952,
  -0.34518,
  -0.249845,

In [9]:
sample_matrix = sp.load_npz('tio2/res/St_ks_1000.npz')

In [10]:
sample_matrix.shape

(64, 64)

In [11]:
# Lowest and highest orbital, Here HOMO is 24
#params['lowest_orbital'] = 24-15
#params['highest_orbital'] = 25+15
# so:

40 - 9 + 1

32

In [12]:
lowest_orbital = 9
ndim = sample_matrix.shape[0] // 2
actual_orbital_space = list(range(lowest_orbital, lowest_orbital + ndim ))

print(actual_orbital_space)

[9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]


## 2. Libra implementation - with active space truncation
<a name="2"></a>[Back to TOC](#toc)


In [13]:
labels, q = CP2K_methods.read_trajectory_xyz_file("tio2/dynamics_AIMD-aligned_100K.xyz", 0)
print(labels)
print(q)

params = {"atom_labels":labels, "timestep":0, 
          "dt":1.0*units.fs2au,
          "nelec_act_space":None,
          "ci_threshold":0.0,
          "lowest_orbital": 9,          
          "nstates": 11,
          "is_first_time":{0:True},
          "act_state":{0:1},
         }
print(params)

# Emulates 1 trajectory
full_id = Py2Cpp_int([0, 0])

res = "tio2_results_1"
# Create working directory, if doesn't exist
if not os.path.exists(res):
    os.mkdir(res)

# Do the first 5 steps 
for i in range(401):
    #print(F"======== Iteration {i} ==============")
    labels, q = CP2K_methods.read_trajectory_xyz_file("tio2/dynamics_AIMD-aligned_100K.xyz", i)
    params["timestep"] = i
    params["logfile_name"] = F"tio2/all_logfiles/step_{1000+i}.log"
    params["time_overlap_filename"] = F"tio2/res/St_ks_{1000+i}.npz"
    
    obj = CP2K_methods.cp2k_compute_adi(q, params, full_id)        
    obj.ham_adi.show_matrix(F"{res}/ham_adi_{i}.txt")
    obj.hvib_adi.show_matrix(F"{res}/hvib_adi_{i}.txt")
    obj.time_overlap_adi.real().show_matrix(F"{res}/st_adi_{i}.txt")

['Ti', 'O', 'O', 'Ti', 'O', 'O']
{'atom_labels': ['Ti', 'O', 'O', 'Ti', 'O', 'O'], 'timestep': 0, 'dt': 41.339396444811904, 'nelec_act_space': None, 'ci_threshold': 0.0, 'lowest_orbital': 9, 'nstates': 11, 'is_first_time': {0: True}, 'act_state': {0: 1}}


/home/alexvakimov/SOFTWARE/libra/_build/src/libra_py/packages/cp2k/methods.py:2929: ComplexWarning: Casting complex values to real discards the imaginary part
  st_mo = st_mo.astype(np.float64, copy=False)
/home/alexvakimov/SOFTWARE/libra/_build/src/libra_py/packages/cp2k/methods.py:2979: ComplexWarning: Casting complex values to real discards the imaginary part
  obj.time_overlap_adi.set(i, j, float(st_ci[i, j]))
